In [0]:
from pyspark.sql.functions import col, to_timestamp, when, abs, upper, round

print("Reading from Bronze layer...")
df_bronze = spark.read.table("crypto_bronze")

print("Applying advanced Silver transformations...")

# 1. Base selection, type casting, and string normalization
df_silver = df_bronze.select(
    col("id").alias("coin_id"),
    upper(col("symbol")).alias("symbol"),  # Make symbols uppercase
    col("name"),
    col("current_price").cast("double"),
    col("market_cap").cast("double"),
    col("high_24h").cast("double"),
    col("low_24h").cast("double"),
    col("price_change_percentage_24h").cast("double"),
    to_timestamp(col("last_updated")).alias("last_updated_timestamp")
)

# 2. Add derived columns using complex PySpark logic
df_silver = df_silver.withColumn(
    # Create Market Cap Tiers
    "market_cap_tier",
    when(col("market_cap") >= 10000000000, "Large Cap (>$10B)")
    .when((col("market_cap") >= 1000000000) & (col("market_cap") < 10000000000), "Mid Cap ($1B - $10B)")
    .otherwise("Small Cap (<$1B)")
).withColumn(
    # Calculate the exact dollar spread between the high and low
    "price_spread_24h",
    round(col("high_24h") - col("low_24h"), 4)
).withColumn(
    # Create a boolean flag for massive price swings (> 10% or < -10%)
    "is_highly_volatile",
    when(abs(col("price_change_percentage_24h")) > 10, True).otherwise(False)
)

# 3. Drop critical nulls (Data validation)
df_silver = df_silver.na.drop(subset=["current_price", "last_updated_timestamp"])

# Save the transformed dataframe to a permanent Delta table
df_silver.write.format("delta").mode("overwrite").saveAsTable("crypto_silver")

print("Successfully saved data to crypto_silver table!")

# Display the enriched data
display(df_silver)

Reading from Bronze layer...
Applying advanced Silver transformations...
Successfully saved data to crypto_silver table!


coin_id,symbol,name,current_price,market_cap,high_24h,low_24h,price_change_percentage_24h,last_updated_timestamp,market_cap_tier,price_spread_24h,is_highly_volatile
bitcoin,BTC,Bitcoin,66179.0,1.327410054155E12,66341.0,64176.0,2.39401,2026-07-21T11:31:06.079Z,Large Cap (>$10B),2165.0,false
ethereum,ETH,Ethereum,1927.84,2.32597149905E11,1946.42,1856.39,2.41051,2026-07-21T11:31:08.491Z,Large Cap (>$10B),90.03,false
tether,USDT,Tether,0.999188,1.84069270073E11,0.999298,0.998846,0.03111,2026-07-21T11:30:52.219Z,Large Cap (>$10B),5.0E-4,false
binancecoin,BNB,BNB,576.19,7.6730958305E10,578.99,565.17,1.31856,2026-07-21T11:31:05.981Z,Large Cap (>$10B),13.82,false
usd-coin,USDC,USDC,0.999847,7.3152008362E10,0.999966,0.999678,0.00628,2026-07-21T11:30:58.079Z,Large Cap (>$10B),3.0E-4,false
ripple,XRP,XRP,1.13,7.0607816954E10,1.14,1.088,3.254,2026-07-21T11:31:02.328Z,Large Cap (>$10B),0.052,false
solana,SOL,Solana,78.15,4.5540555686E10,78.64,75.95,2.29828,2026-07-21T11:30:55.195Z,Large Cap (>$10B),2.69,false
tron,TRX,TRON,0.327386,3.1060553984E10,0.327364,0.325059,0.48987,2026-07-21T11:31:04.559Z,Large Cap (>$10B),0.0023,false
figure-heloc,FIGR_HELOC,Figure Heloc,1.001,2.012396333E10,1.018,1.0,-1.68261,2026-07-21T07:23:21.494Z,Large Cap (>$10B),0.018,false
hyperliquid,HYPE,Hyperliquid,62.74,1.3955826394E10,63.3,60.58,3.40951,2026-07-21T11:31:09.416Z,Large Cap (>$10B),2.72,false
